# 26 — Gold Point-in-Time Audit

**Objetivo:** demostrar que el dataset final de modelado puede reconstruirse como si estuviéramos ubicados en cada `t0`, sin información futura.

Este notebook materializa `data/gold/ml/argos_v1.parquet` únicamente después de pasar las auditorías de grain, labels, leakage y timestamps.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.pit_audit import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

'm3-26-pit-audit-v1'

In [2]:
from argos.gold.pit_audit import build_pit_audit

result = build_pit_audit(
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

str(result.artifact.relative_to(paths.root))

[26] materializing audited model matrix from Gold feature components...


'data/gold/ml/argos_v1.parquet'

## Feature registry auditado

In [3]:
result.feature_registry

,feature,source,point_in_time_policy,model_role
0,t0_year,contract_features,conservative signing-time whitelist,feature
1,t0_month,contract_features,conservative signing-time whitelist,feature
2,t0_day_of_week,contract_features,conservative signing-time whitelist,feature
3,t0_quarter,contract_features,conservative signing-time whitelist,feature
4,departamento,contract_features,conservative signing-time whitelist,feature
5,ciudad,contract_features,conservative signing-time whitelist,feature
6,orden,contract_features,conservative signing-time whitelist,feature
7,sector,contract_features,conservative signing-time whitelist,feature
8,rama,contract_features,conservative signing-time whitelist,feature
9,entidad_centralizada,contract_features,conservative signing-time whitelist,feature


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,model_matrix_grain,grain,True,True,"{""distinct_contracts"": 4783128, ""expected_assi...",one model-matrix row per assigned contract
1,model_matrix_labels_complete,target,True,True,"{""null_labels"": 0}",train/validation/test model rows all have bina...
2,gold_t0_alignment,point_in_time,True,True,"{""t0_mismatches"": 0}",all feature components use the same contract t0
3,historical_pit_audit,point_in_time,True,True,"{""future_or_same_time_violations"": 0}",historical features use only timestamps strict...
4,competition_pit_audit,point_in_time,True,True,"{""future_publication_violations"": 0}",competition features use no bidder observation...
5,forbidden_snapshot_fields_absent,leakage,True,True,"{""forbidden_present"": []}",known mutable/post-t0 Silver snapshot fields a...
6,pit_audit_metadata_not_features,leakage,True,True,"{""pit_metadata_present"": []}",PIT audit timestamps are not exposed as model ...
7,feature_registry_nonempty,features,True,True,"{""feature_count"": 41}",audited model feature registry is non-empty an...


In [5]:
result.issue_register

,check_id,severity,observed,requirement,action


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("26 ✅ PASS" if result.ready_for_gate else "26 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_gate, "Gold PIT audit permanece OPEN; no ejecutar gate M3."

26 ✅ PASS
Blocking checks: 8/8
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/gold_pit_audit.json
